# GlanceDesk — Even Realities G2 Real-Time Voice-to-Trade & Telegram GPU Server

**1-Click Google Colab GPU Runtime (`A100` / `L4` / `T4`)** by **Upswing-Labs** for **Heads Up: Even Realities Hackathon (DoraHacks)**.

### What this notebook runs (`< 350 ms` target end-to-end)
1. **`Faster-Whisper large-v3-turbo`** (`float16` on CUDA GPU, `int8` fallback on CPU) with pre-warmed CTranslate2 kernels.
2. **Silero VAD** voice-activity trimming (eliminates dead silence from Even G2's 16 kHz mono s16le PCM mic stream).
3. **Deterministic Structured Intent Parser + Safety Guard** (`BUY_POLYMARKET`, `PLACE_HYPERLIQUID`, `REPLY_TELEGRAM`, `SUMMARIZE_ALERTS`, `CANCEL`) enforcing **TESTNET / PAPER ONLY** (`$50` hard cap, blocking mainnet/withdrawal commands) and emitting `<= 4` lines (`<= 36` chars/line) formatted for the **576×288 4-bit green G2 waveguide**.
4. **FastAPI + HTTPS `cloudflared` Quick Tunnel** exposing `/health` and `/v1/voice-intent` (raw 16 kHz PCM, WAV/WebM upload, or JSON).

> **How to run:** `Runtime -> Change runtime type -> T4 / L4 / A100 GPU`, then click **`Runtime -> Run all`**. Copy the printed `https://*.trycloudflare.com` URL and bind it to your Cloudflare Worker with `wrangler secret put COLAB_WHISPER_URL`.


## Cell 1 — Configuration & GPU Hardware Auto-Tuning (`A100` / `L4` / `T4`)
Detects CUDA GPU architecture, selects optimal `float16` compute settings, and sets safety guardrails.


In [ ]:
# @title GlanceDesk GPU Whisper Server Configuration
import os
import platform
import subprocess
import sys

# ---- User Controls (Colab Form or direct edit) ----
WHISPER_MODEL = "large-v3-turbo"  # @param ["large-v3-turbo", "distil-large-v3", "small.en"]
SERVER_PORT = 8000                # @param {type:"integer"}
ENABLE_VAD = True                 # @param {type:"boolean"}
ENABLE_CLOUDFLARED = True         # @param {type:"boolean"}
MAX_PAPER_TRADE_USD = 50.0        # @param {type:"number"}
DEFAULT_PAPER_EQUITY_USD = 2500.0 # @param {type:"number"}
AUTHOR_ORG = "Upswing-Labs"

# Optional shared header secret (leave empty for open hackathon demo; set in Colab env for lock-down)
GLANCEDESK_API_TOKEN = os.environ.get("GLANCEDESK_API_TOKEN", "").strip()

def detect_gpu_profile() -> dict:
    profile = {
        "python": platform.python_version(),
        "device": "cpu",
        "compute_type": "int8",
        "gpu_name": "CPU-only",
        "vram_gb": 0.0,
        "beam_size": 1,
    }
    try:
        import torch  # noqa: PLC0415
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            vram_gb = round(props.total_memory / (1024 ** 3), 2)
            profile.update(
                {
                    "device": "cuda",
                    "compute_type": "float16",
                    "gpu_name": props.name,
                    "vram_gb": vram_gb,
                    "beam_size": 1,
                }
            )
    except Exception as exc:
        profile["torch_note"] = str(exc)
    return profile

GPU_PROFILE = detect_gpu_profile()
print(f"[{AUTHOR_ORG}] GlanceDesk Hardware Profile:", GPU_PROFILE)


## Cell 2 — Install Runtime Dependencies & `cloudflared` (Safe Direct Binary)
Installs `faster-whisper`, `silero-vad`, `fastapi`, `uvicorn`, `python-multipart`, and `scipy` cleanly via `sys.executable -m pip`, plus installs the official Cloudflare `cloudflared` `.deb` package on Linux (without piping scripts to shell).


In [ ]:
# @title Install Faster-Whisper, Silero VAD, FastAPI, and cloudflared
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

PACKAGES = [
    "faster-whisper>=1.1.0",
    "silero-vad>=5.1.2",
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.32.0",
    "python-multipart>=0.0.12",
    "numpy>=1.26.0",
    "scipy>=1.13.0",
]

print("Installing Python packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES],
    check=True,
)

def ensure_cloudflared() -> str | None:
    existing = shutil.which("cloudflared")
    if existing:
        print("cloudflared already installed:", existing)
        return existing
    if sys.platform != "linux":
        print("Skipping cloudflared .deb install on non-Linux platform:", sys.platform)
        return None
    deb_path = Path("/tmp/cloudflared-linux-amd64.deb")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    print("Downloading cloudflared package from GitHub releases...")
    urllib.request.urlretrieve(url, deb_path)
    subprocess.run(["dpkg", "-i", str(deb_path)], check=True)
    found = shutil.which("cloudflared")
    print("cloudflared ready at:", found)
    return found

CLOUDFLARED_BIN = ensure_cloudflared() if ENABLE_CLOUDFLARED else None
print("Dependency setup complete.")


## Cell 3 — Load `Faster-Whisper large-v3-turbo` (`float16` GPU) + Silero VAD + Kernel Warmup
Loads the CTranslate2 GPU model once into VRAM, initializes Silero VAD with an automatic RMS energy fallback, and runs a 0.5 s warmup pass so the first Even G2 voice command hits `< 350 ms` latency.


In [ ]:
# @title Load Faster-Whisper large-v3-turbo & Silero VAD
import io
import time
import wave
import numpy as np
from faster_whisper import WhisperModel

t_load = time.perf_counter()
whisper_model = WhisperModel(
    WHISPER_MODEL,
    device=GPU_PROFILE["device"],
    compute_type=GPU_PROFILE["compute_type"],
)
model_load_s = round(time.perf_counter() - t_load, 2)

vad_model = None
get_speech_timestamps = None
if ENABLE_VAD:
    try:
        import torch  # noqa: PLC0415
        from silero_vad import get_speech_timestamps as _gst, load_silero_vad  # noqa: PLC0415
        vad_model = load_silero_vad()
        get_speech_timestamps = _gst
        print("Silero VAD loaded cleanly.")
    except Exception as exc:
        print(f"Silero VAD unavailable ({exc}); using deterministic RMS energy gate fallback.")

# Warmup inference on 0.5s silence so CUDA/CTranslate2 kernels are hot
warmup_audio = np.zeros(8000, dtype=np.float32)
t_warm = time.perf_counter()
_segs, _info = whisper_model.transcribe(
    warmup_audio,
    language="en",
    beam_size=1,
    vad_filter=False,
)
_ = list(_segs)
warmup_ms = round((time.perf_counter() - t_warm) * 1000, 1)
print(
    f"Loaded {WHISPER_MODEL} ({GPU_PROFILE['device']}/{GPU_PROFILE['compute_type']}) "
    f"in {model_load_s}s | warmup={warmup_ms}ms"
)

def trim_with_vad(audio_f32: np.ndarray, sample_rate: int = 16000) -> tuple[np.ndarray, int]:
    """Trims leading/trailing silence using Silero VAD or RMS energy gate."""
    if audio_f32.size == 0:
        return audio_f32, 0
    if vad_model is not None and get_speech_timestamps is not None and sample_rate == 16000:
        try:
            import torch  # noqa: PLC0415
            tensor = torch.from_numpy(audio_f32)
            stamps = get_speech_timestamps(tensor, vad_model, sampling_rate=sample_rate)
            if stamps:
                start = max(0, int(stamps[0]["start"]) - 1600)
                end = min(audio_f32.size, int(stamps[-1]["end"]) + 1600)
                trimmed = audio_f32[start:end]
                speech_ms = int(round((trimmed.size / sample_rate) * 1000))
                return trimmed, speech_ms
        except Exception:
            pass
    # Deterministic RMS energy gate fallback
    frame = max(1, int(sample_rate * 0.02))
    trimmed = audio_f32
    if audio_f32.size > frame * 4:
        energies = np.sqrt(
            np.mean(
                audio_f32[: (audio_f32.size // frame) * frame].reshape(-1, frame) ** 2,
                axis=1,
            )
        )
        active = np.where(energies > 0.008)[0]
        if active.size > 0:
            s = max(0, int(active[0]) * frame - frame * 2)
            e = min(audio_f32.size, (int(active[-1]) + 1) * frame + frame * 2)
            trimmed = audio_f32[s:e]
    speech_ms = int(round((trimmed.size / sample_rate) * 1000))
    return trimmed, speech_ms


## Cell 4 — Structured Voice-to-Trade & Telegram Intent Parser + 576×288 Safety HUD Formatter
Parses Whisper transcripts into strict JSON (`BUY_POLYMARKET`, `PLACE_HYPERLIQUID`, `REPLY_TELEGRAM`, `SUMMARIZE_ALERTS`, `CANCEL`), enforces `$50` testnet/paper safety caps, blocks mainnet/withdrawal keywords, and formats `<= 4` concise lines (`<= 36` chars/line) for the G2 display.


In [ ]:
# @title Deterministic Structured Intent Parser & G2 576x288 HUD Formatter
import re
from typing import Any

MAX_HUD_LINES = 4
MAX_LINE_CHARS = 36

SYMBOL_ALIASES = {
    "bitcoin": "BTC",
    "btc": "BTC",
    "ethereum": "ETH",
    "ether": "ETH",
    "eth": "ETH",
    "solana": "SOL",
    "sol": "SOL",
    "hyperliquid": "HYPE",
    "hype": "HYPE",
    "toncoin": "TON",
    "ton": "TON",
    "avalanche": "AVAX",
    "avax": "AVAX",
    "dogecoin": "DOGE",
    "doge": "DOGE",
    "arbitrum": "ARB",
    "arb": "ARB",
    "sui": "SUI",
    "bnb": "BNB",
}

FORBIDDEN_PATTERNS = [
    re.compile(r"\bmainnet\b", re.IGNORECASE),
    re.compile(r"\breal\s+(?:money|funds|usdt|usdc)\b", re.IGNORECASE),
    re.compile(r"\bwithdraw(?:al)?\b", re.IGNORECASE),
    re.compile(r"\btransfer\s+(?:all|everything|wallet)\b", re.IGNORECASE),
    re.compile(r"\b(?:seed\s*phrase|private\s*key|mnemonic|export\s+key)\b", re.IGNORECASE),
    re.compile(r"\bdrain\b", re.IGNORECASE),
]

def sanitize_hud(text: str) -> str:
    cleaned = re.sub(r"[\r\n\t]+", " ", str(text or ""))
    cleaned = re.sub(r"[^\x20-\x7E\u00B7]", "", cleaned)
    return re.sub(r"\s{2,}", " ", cleaned).strip()

def clamp_line(text: str, max_chars: int = MAX_LINE_CHARS) -> str:
    s = sanitize_hud(text)
    if len(s) <= max_chars:
        return s
    return s[: max_chars - 2].rstrip() + ".."

def normalize_hud_lines(lines: list[str]) -> list[str]:
    out = [clamp_line(x) for x in lines[:MAX_HUD_LINES] if sanitize_hud(x)]
    while len(out) < MAX_HUD_LINES:
        out.append("[TAP=CONFIRM] [HOLD=ABORT]")
    return out[:MAX_HUD_LINES]

def extract_usd_size(text: str, default: float = 25.0) -> float:
    m = re.search(r"\$\s*(\d+(?:\.\d{1,2})?)", text)
    if m:
        return float(m.group(1))
    m = re.search(r"\b(\d+(?:\.\d{1,2})?)\s*(?:usd|usdc|usdt|dollars?|bucks?)\b", text, re.IGNORECASE)
    if m:
        return float(m.group(1))
    m = re.search(r"\b(?:for|size|of|with|bet|buy|short|long|sell)\s+(?:[a-z]+\s+){0,2}(\d+(?:\.\d{1,2})?)\b", text, re.IGNORECASE)
    if m:
        return float(m.group(1))
    return default

def resolve_symbol(text: str) -> str | None:
    lower = text.lower()
    for alias, sym in SYMBOL_ALIASES.items():
        if re.search(rf"\b{re.escape(alias)}\b", lower):
            return sym
    return None

def parse_structured_intent(transcript: str, confidence: float = 0.95) -> dict[str, Any]:
    clean = sanitize_hud(transcript)
    lower = clean.lower()

    if not clean:
        return {
            "ok": False,
            "transcript": "",
            "action": "UNKNOWN",
            "symbol_or_market": None,
            "side": None,
            "size_usd": 0.0,
            "target_chat": None,
            "reply_text": None,
            "confidence": 0.0,
            "requires_confirmation": False,
            "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
            "hud_confirmation_lines": normalize_hud_lines([
                "NO SPEECH DETECTED",
                "Hold steady and retry",
                "Say: Short ETH $25 / Reply TG",
                "[TAP=RETRY] [HOLD=FEED]",
            ]),
        }

    for pat in FORBIDDEN_PATTERNS:
        if pat.search(clean):
            return {
                "ok": False,
                "transcript": clean,
                "action": "BLOCKED_SAFETY",
                "symbol_or_market": None,
                "side": None,
                "size_usd": 0.0,
                "target_chat": None,
                "reply_text": None,
                "confidence": 1.0,
                "requires_confirmation": False,
                "reason": "Blocked: GlanceDesk is TESTNET/PAPER ONLY (no mainnet or withdrawals).",
                "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
                "hud_confirmation_lines": normalize_hud_lines([
                    "BLOCKED · SAFETY GUARD",
                    "Mainnet / withdraw denied",
                    f"Testnet paper cap: ${int(MAX_PAPER_TRADE_USD)} max",
                    "[TAP=RETURN TO FEED]",
                ]),
            }

    if re.match(r"^(?:cancel|abort|stop|nevermind|never\s+mind|stand\s+down|close)\b", lower):
        return {
            "ok": True,
            "transcript": clean,
            "action": "CANCEL",
            "symbol_or_market": None,
            "side": None,
            "size_usd": 0.0,
            "target_chat": None,
            "reply_text": None,
            "confidence": 0.99,
            "requires_confirmation": False,
            "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
            "hud_confirmation_lines": normalize_hud_lines([
                "ACTION ABORTED",
                "Returned to Glance Feed",
                "Zero testnet orders placed",
                "[TAP=MIC] [RING=SCROLL]",
            ]),
        }

    tg_m = (
        re.match(r"^(?:reply|respond)\s+(?:to\s+)?(?:telegram\s+)?([^:,-]{1,28}?)\s*(?::|,|-|\bsaying\b|\bthat\b)\s*(.+)$", clean, re.IGNORECASE)
        or re.match(r"^(?:tell|message|text|ping|dm)\s+(?:telegram\s+)?([^:,-]{1,28}?)\s*(?::|,|-|\bsaying\b|\bthat\b|\s)\s*(.+)$", clean, re.IGNORECASE)
        or re.match(r"^(?:send\s+(?:to\s+)?telegram|telegram\s+reply|tg\s+reply)\s*(?::|,|-)?\s*(.+)$", clean, re.IGNORECASE)
    )
    if tg_m:
        if len(tg_m.groups()) == 2:
            target_chat = sanitize_hud(tg_m.group(1)) or "Alpha Desk"
            reply_body = sanitize_hud(tg_m.group(2))
        else:
            target_chat = "Alpha Desk"
            reply_body = sanitize_hud(tg_m.group(1))
        return {
            "ok": True,
            "transcript": clean,
            "action": "REPLY_TELEGRAM",
            "symbol_or_market": "TELEGRAM",
            "side": "SEND",
            "size_usd": 0.0,
            "target_chat": target_chat,
            "reply_text": reply_body,
            "confidence": round(confidence, 3),
            "requires_confirmation": True,
            "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
            "hud_confirmation_lines": normalize_hud_lines([
                f"SEND TG -> {clamp_line(target_chat, 18)}?",
                f'"{clamp_line(reply_body, 32)}"',
                f"Conf {int(confidence * 100)}% · 1-Tap Voice Reply",
                "[TAP=SEND TG] [HOLD=ABORT]",
            ]),
        }

    if re.search(r"\b(?:summarize|summary|digest|alerts?|what\s+moved|brief\s+me|status|read\s+telegram)\b", lower):
        return {
            "ok": True,
            "transcript": clean,
            "action": "SUMMARIZE_ALERTS",
            "symbol_or_market": "ALL_FEEDS",
            "side": "READ",
            "size_usd": 0.0,
            "target_chat": None,
            "reply_text": None,
            "confidence": round(confidence, 3),
            "requires_confirmation": False,
            "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
            "hud_confirmation_lines": normalize_hud_lines([
                "ALERT DIGEST READY",
                "Loading Polymarket + HL + TG",
                "Mode: TESTNET / PAPER",
                "[TAP=FEED] [RING=SCROLL]",
            ]),
        }

    is_pm = bool(
        re.search(r"\b(?:polymarket|bet|wager|odds|prediction|forecast)\b", lower)
        or re.search(r"\b(?:buy|take|put)\b.*\b(?:yes|no)\b", lower)
    )
    if is_pm:
        side = "NO" if re.search(r"\bno\b", lower) else "YES"
        req_usd = extract_usd_size(clean, 15.0)
        capped = req_usd > MAX_PAPER_TRADE_USD
        size_usd = round(max(1.0, min(req_usd, MAX_PAPER_TRADE_USD)), 2)
        market_label = "BTC > $100k EOY" if "btc" in lower or "bitcoin" in lower else "Fed 25bp Cut" if "fed" in lower or "rate" in lower else "Top Polymarket"
        return {
            "ok": True,
            "transcript": clean,
            "action": "BUY_POLYMARKET",
            "symbol_or_market": market_label,
            "side": side,
            "size_usd": size_usd,
            "target_chat": None,
            "reply_text": None,
            "confidence": round(confidence, 3),
            "requires_confirmation": True,
            "safety_caps": {
                "mode": "TESTNET_PAPER_ONLY",
                "max_usd": MAX_PAPER_TRADE_USD,
                "requested_usd": req_usd,
                "capped": capped,
            },
            "hud_confirmation_lines": normalize_hud_lines([
                f"BET {side} ${int(size_usd)} PAPER?",
                f"{clamp_line(market_label, 22)} · LIVE",
                f"Polymarket Paper · Max ${int(MAX_PAPER_TRADE_USD)}",
                "[TAP=CONFIRM] [HOLD=ABORT]",
            ]),
        }

    sym = resolve_symbol(clean)
    is_short = bool(re.search(r"\b(?:short|sell|fade|hedge)\b", lower))
    is_long = bool(re.search(r"\b(?:long|buy|bid)\b", lower))
    if sym and (is_short or is_long):
        side = "SHORT" if is_short else "LONG"
        req_usd = extract_usd_size(clean, 25.0)
        capped = req_usd > MAX_PAPER_TRADE_USD
        size_usd = round(max(1.0, min(req_usd, MAX_PAPER_TRADE_USD)), 2)
        risk_pct = round((size_usd / DEFAULT_PAPER_EQUITY_USD) * 100.0, 1)
        return {
            "ok": True,
            "transcript": clean,
            "action": "PLACE_HYPERLIQUID",
            "symbol_or_market": sym,
            "side": side,
            "size_usd": size_usd,
            "target_chat": None,
            "reply_text": None,
            "confidence": round(confidence, 3),
            "requires_confirmation": True,
            "safety_caps": {
                "mode": "TESTNET_PAPER_ONLY",
                "max_usd": MAX_PAPER_TRADE_USD,
                "requested_usd": req_usd,
                "capped": capped,
            },
            "hud_confirmation_lines": normalize_hud_lines([
                f"{side} {sym} ${int(size_usd)} TESTNET?",
                f"Entry LIVE MID · Risk {risk_pct}%",
                f"Voice {int(confidence * 100)}% · Max ${int(MAX_PAPER_TRADE_USD)} cap",
                "[TAP=CONFIRM] [HOLD=ABORT]",
            ]),
        }

    return {
        "ok": False,
        "transcript": clean,
        "action": "UNKNOWN",
        "symbol_or_market": sym,
        "side": None,
        "size_usd": 0.0,
        "target_chat": None,
        "reply_text": None,
        "confidence": 0.25,
        "requires_confirmation": False,
        "safety_caps": {"mode": "TESTNET_PAPER_ONLY", "max_usd": MAX_PAPER_TRADE_USD, "capped": False},
        "hud_confirmation_lines": normalize_hud_lines([
            "COMMAND NOT RECOGNIZED",
            clamp_line(clean, 34),
            "Say: Short ETH $25 / Reply TG",
            "[TAP=RETRY] [HOLD=FEED]",
        ]),
    }

print("Intent parser self-test:", parse_structured_intent("Short ETH 25 dollars")["hud_confirmation_lines"])


## Cell 5 — FastAPI Voice-to-Intent Server (`/health` & `/v1/voice-intent`)
Accepts:
1. Raw `audio/pcm` / `application/octet-stream` (16 kHz mono 16-bit signed LE PCM from Even G2's 4-mic array).
2. `multipart/form-data` WAV / WebM / OGG / MP3 uploads (decoded cleanly with `scipy` / `ffmpeg`).
3. `application/json` with `{ "audio_pcm_b64": "...", "sample_rate": 16000 }` or `{ "text": "..." }`.


In [ ]:
# @title Build FastAPI Real-Time Voice-to-Intent App
import base64
import io
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Any

import numpy as np
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from scipy.io import wavfile
from scipy.signal import resample_poly

app = FastAPI(title="GlanceDesk GPU Whisper Intent Server", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def decode_audio_to_16k_f32(raw_bytes: bytes, content_type: str, sample_rate: int = 16000) -> np.ndarray:
    if not raw_bytes:
        return np.zeros(0, dtype=np.float32)

    ctype = (content_type or "").lower()
    # Raw 16-bit signed little-endian mono PCM (Even Realities G2 native format)
    if "audio/pcm" in ctype or "application/octet-stream" in ctype or not ctype:
        usable = len(raw_bytes) - (len(raw_bytes) % 2)
        pcm_i16 = np.frombuffer(raw_bytes[:usable], dtype="<i2")
        audio = pcm_i16.astype(np.float32) / 32768.0
        if sample_rate != 16000 and sample_rate > 0:
            audio = resample_poly(audio, 16000, sample_rate).astype(np.float32)
        return audio

    # WAV payload
    if "wav" in ctype or raw_bytes[:4] == b"RIFF":
        sr, data = wavfile.read(io.BytesIO(raw_bytes))
        if data.ndim > 1:
            data = data.mean(axis=1)
        if np.issubdtype(data.dtype, np.integer):
            max_val = float(np.iinfo(data.dtype).max)
            audio = data.astype(np.float32) / max_val
        else:
            audio = data.astype(np.float32)
        if sr != 16000 and sr > 0:
            audio = resample_poly(audio, 16000, sr).astype(np.float32)
        return audio

    # WebM / OGG / MP3 / M4A via ffmpeg to 16kHz mono s16le PCM
    with tempfile.NamedTemporaryFile(suffix=".bin", delete=True) as tmp_in:
        tmp_in.write(raw_bytes)
        tmp_in.flush()
        proc = subprocess.run(
            [
                "ffmpeg",
                "-nostdin",
                "-y",
                "-i",
                tmp_in.name,
                "-ac",
                "1",
                "-ar",
                "16000",
                "-f",
                "s16le",
                "pipe:1",
            ],
            capture_output=True,
            check=False,
        )
        if proc.returncode == 0 and proc.stdout:
            pcm_i16 = np.frombuffer(proc.stdout, dtype="<i2")
            return pcm_i16.astype(np.float32) / 32768.0
    return np.zeros(0, dtype=np.float32)

def transcribe_f32(audio_f32: np.ndarray) -> tuple[str, float, int, int]:
    trimmed, speech_ms = trim_with_vad(audio_f32, 16000)
    if trimmed.size < 800:
        return "", 0.0, 0, speech_ms
    t0 = time.perf_counter()
    segments, info = whisper_model.transcribe(
        trimmed,
        language="en",
        beam_size=GPU_PROFILE["beam_size"],
        best_of=1,
        temperature=0.0,
        condition_on_previous_text=False,
        vad_filter=False,
        initial_prompt="Short ETH 25 dollars, Long BTC 40 dollars, Bet 15 dollars YES Polymarket, Reply to Alpha Desk on Telegram.",
    )
    texts = []
    probs = []
    for seg in segments:
        txt = (seg.text or "").strip()
        if txt:
            texts.append(txt)
            probs.append(max(0.5, min(0.99, float(np.exp(getattr(seg, "avg_logprob", -0.1))))))
    asr_ms = int(round((time.perf_counter() - t0) * 1000))
    transcript = " ".join(texts).strip()
    conf = float(np.mean(probs)) if probs else float(getattr(info, "language_probability", 0.9))
    return transcript, conf, asr_ms, speech_ms

@app.get("/health")
def health() -> dict[str, Any]:
    return {
        "ok": True,
        "service": "glancedesk-colab-whisper",
        "org": AUTHOR_ORG,
        "model": WHISPER_MODEL,
        "gpu": GPU_PROFILE["gpu_name"],
        "vram_gb": GPU_PROFILE["vram_gb"],
        "device": GPU_PROFILE["device"],
        "compute_type": GPU_PROFILE["compute_type"],
        "vad_ready": vad_model is not None,
        "warmup_ms": warmup_ms,
        "execution_mode": "TESTNET_PAPER_ONLY",
        "max_paper_trade_usd": MAX_PAPER_TRADE_USD,
    }

@app.post("/v1/voice-intent")
async def voice_intent(
    request: Request,
    x_sample_rate: int = Header(default=16000),
    x_glancedesk_token: str = Header(default=""),
) -> dict[str, Any]:
    if GLANCEDESK_API_TOKEN and x_glancedesk_token != GLANCEDESK_API_TOKEN:
        raise HTTPException(status_code=401, detail="Unauthorized GlanceDesk token")

    t_start = time.perf_counter()
    ctype = (request.headers.get("content-type") or "").lower()
    transcript = ""
    conf = 0.95
    asr_ms = 0
    vad_speech_ms = 0

    if "application/json" in ctype:
        payload = await request.json()
        if isinstance(payload, dict) and payload.get("audio_pcm_b64"):
            raw = base64.b64decode(payload["audio_pcm_b64"])
            sr = int(payload.get("sample_rate") or x_sample_rate or 16000)
            audio = decode_audio_to_16k_f32(raw, "audio/pcm", sr)
            transcript, conf, asr_ms, vad_speech_ms = transcribe_f32(audio)
        elif isinstance(payload, dict):
            transcript = str(payload.get("text") or payload.get("transcript") or "").strip()
            conf = 0.97
    elif "multipart/form-data" in ctype:
        form = await request.form()
        upload = form.get("file") or form.get("audio")
        if upload is not None and hasattr(upload, "read"):
            raw = await upload.read()
            upload_ct = getattr(upload, "content_type", "") or "audio/wav"
            audio = decode_audio_to_16k_f32(raw, upload_ct, x_sample_rate)
            transcript, conf, asr_ms, vad_speech_ms = transcribe_f32(audio)
        elif form.get("text"):
            transcript = str(form.get("text")).strip()
    else:
        raw = await request.body()
        audio = decode_audio_to_16k_f32(raw, ctype or "audio/pcm", x_sample_rate)
        transcript, conf, asr_ms, vad_speech_ms = transcribe_f32(audio)

    result = parse_structured_intent(transcript, confidence=conf)
    result["asr_ms"] = asr_ms
    result["vad_speech_ms"] = vad_speech_ms
    result["latency_ms"] = int(round((time.perf_counter() - t_start) * 1000))
    result["engine"] = f"faster-whisper-{WHISPER_MODEL}-{GPU_PROFILE['compute_type']}"
    return result

print("FastAPI endpoints registered: GET /health, POST /v1/voice-intent")


## Cell 6 — Launch Uvicorn + HTTPS `cloudflared` Quick Tunnel & Self-Test
Starts FastAPI in a background thread, opens a zero-config `https://*.trycloudflare.com` tunnel, runs live `/health` + `/v1/voice-intent` verification, and prints the exact `wrangler secret put COLAB_WHISPER_URL` command.


In [ ]:
# @title Start GlanceDesk GPU Whisper Server & Cloudflare Quick Tunnel
import json
import re
import subprocess
import threading
import time
import urllib.request
import uvicorn

def run_uvicorn() -> None:
    uvicorn.run(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2.0)

# Local health & latency verification
with urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/health", timeout=5) as resp:
    health_data = json.loads(resp.read().decode("utf-8"))
print("Local /health OK:", json.dumps(health_data, indent=2))

test_req = urllib.request.Request(
    f"http://127.0.0.1:{SERVER_PORT}/v1/voice-intent",
    data=json.dumps({"text": "Short ETH 25 dollars"}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(test_req, timeout=10) as resp:
    sample_intent = json.loads(resp.read().decode("utf-8"))
print("Local /v1/voice-intent OK:", json.dumps(sample_intent, indent=2))

PUBLIC_TUNNEL_URL = None
tunnel_proc = None

if ENABLE_CLOUDFLARED and CLOUDFLARED_BIN:
    tunnel_proc = subprocess.Popen(
        [
            CLOUDFLARED_BIN,
            "tunnel",
            "--url",
            f"http://127.0.0.1:{SERVER_PORT}",
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    deadline = time.time() + 25.0
    url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    while time.time() < deadline and tunnel_proc.stdout is not None:
        line = tunnel_proc.stdout.readline()
        if not line:
            break
        match = url_re.search(line)
        if match:
            PUBLIC_TUNNEL_URL = match.group(0)
            break

if PUBLIC_TUNNEL_URL:
    print("=" * 72)
    print("GLANCEDESK COLAB GPU WHISPER TUNNEL LIVE:")
    print(f"  Health URL : {PUBLIC_TUNNEL_URL}/health")
    print(f"  Voice URL  : {PUBLIC_TUNNEL_URL}/v1/voice-intent")
    print("")
    print("Bind to your Cloudflare Worker:")
    print(f"  echo {PUBLIC_TUNNEL_URL} | npx wrangler secret put COLAB_WHISPER_URL")
    print("=" * 72)
else:
    print(f"Server running locally at http://127.0.0.1:{SERVER_PORT} (tunnel disabled or pending).")
